# E12：多表合并、结果检查与导出

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用 `concat()` 纵向拼接同结构数据。
2. 根据数据结构选择单键或多键完成 `merge()`。
3. 比较 `inner` 与 `left` 连接，并区分未匹配记录和匹配后的数值缺失。
4. 复用分组汇总能力构造学生层指标。
5. 核验并导出一人一行的分析就绪表。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查行数、字段、主键和缺失值口径。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的路径和输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- DataFrame 结果异常时，先检查 `shape`、字段名、数据类型和缺失值。
- 如果变量状态混乱，重启内核并从上到下重新运行。
- 按 `TODO` 补全待写代码，再取消相关行的注释。标为“课程提供”的准备代码可直接运行；核验代码按单元中的提示运行。

## 建议节奏

- 输入选择与核验：10 分钟
- 纵向拼接与项目表检查：10 分钟
- `inner`、`left` 与匹配检查：20 分钟
- 学生层汇总：20 分钟
- 多表合并：15 分钟
- 最终核验、说明与提交：15 分钟

## 主线输入：个人成果优先、官方检查点兜底

E12 同时使用 E10 的学生调查主表和 E11 的四轮学习记录。输入准备代码优先读取个人成果；文件不存在时使用官方检查点。个人成果一旦被选中但核验失败，程序会停止，不会静默替换为官方答案。

找不到个人成果时，按仓库官方检查点、当前工作目录中的 `_official.csv` 副本、联网下载的顺序查找。下载的检查点通过核验后才缓存。项目成绩和课程结果表也优先使用本地文件，缺失时下载到当前工作目录。

In [ ]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

# 有可靠的个人成果时保持 True；需要从官方检查点重新开始时改为 False。
USE_STUDENT_E10_OUTPUT = True
USE_STUDENT_E11_OUTPUT = True

chapter4_dir = Path(".").resolve()
notebook_names = ["e12.ipynb", "e12_inclass.ipynb"]
if not any((chapter4_dir / name).exists() for name in notebook_names):
    chapter4_dir = Path("lab/applied-stat-64/chapter4").resolve()

shared_dir = chapter4_dir.parents[1] / "shared"
student_e10_path = (
    chapter4_dir / "e10_output" / "e10_students_clean.csv"
)
student_e11_path = (
    chapter4_dir
    / "e11_output"
    / "e11_learning_records_clean.csv"
)
official_e10_path = (
    shared_dir / "checkpoints" / "e10_students_clean.csv"
)
official_e11_path = (
    shared_dir
    / "checkpoints"
    / "e11_learning_records_clean.csv"
)
project_path = shared_dir / "raw" / "project_scores_raw.csv"
outcome_path = shared_dir / "raw" / "course_outcomes_raw.csv"
output_path = (
    chapter4_dir / "e12_output" / "e12_analysis_ready.csv"
)


downloaded_checkpoints = {}


def save_download(cache_path, data):
    try:
        cache_path.write_bytes(data)
    except OSError as error:
        raise RuntimeError(
            f"课程数据已下载，但无法保存到 {cache_path}。"
            "请检查当前工作目录的写入权限或磁盘空间。"
        ) from error


def local_or_download(path, cache_name, *, defer_cache=False):
    if path.exists():
        return path

    cache_path = Path(cache_name).resolve()
    if cache_path.exists():
        print(f"使用当前工作目录中的课程数据：{cache_path}")
        return cache_path

    url = (
        "https://rwang0040.github.io/python-for-statistics-public/"
        f"lab/shared/{path.relative_to(shared_dir).as_posix()}"
    )
    print(f"本地未找到课程数据，尝试联网获取：{url}")
    try:
        with urlopen(url, timeout=10) as response:
            data = response.read()
    except OSError as error:
        raise RuntimeError(
            f"无法联网获取课程数据。请检查网络，或下载 {url}，"
            f"保存为 {cache_name} 并放到当前工作目录。"
        ) from error
    if defer_cache:
        downloaded_checkpoints[cache_path] = data
        return BytesIO(data)

    save_download(cache_path, data)
    return cache_path


def choose_input(use_student, student_path, official_path, label):
    if use_student and student_path.exists():
        print(f"{label} 输入来源：学生个人成果")
        return student_path
    if use_student:
        print(f"未找到 {label} 个人成果，改用官方检查点。")
    print(f"{label} 输入来源：官方检查点")
    cache_name = f"{official_path.stem}_official{official_path.suffix}"
    return local_or_download(official_path, cache_name, defer_cache=True)


def numeric_range_ok(frame, column, lower, upper):
    return (
        pd.api.types.is_numeric_dtype(frame[column])
        and frame[column].dropna().between(lower, upper).all()
    )


students_path = choose_input(
    USE_STUDENT_E10_OUTPUT,
    student_e10_path,
    official_e10_path,
    "E10",
)
learning_path = choose_input(
    USE_STUDENT_E11_OUTPUT,
    student_e11_path,
    official_e11_path,
    "E11",
)

students = pd.read_csv(students_path)
learning = pd.read_csv(learning_path)

expected_student_columns = [
    "student_id",
    "alias",
    "gender",
    "major",
    "prior_programming",
    "support_plan",
    "reported_study_hours",
    "reported_attendance_rate",
]
expected_learning_columns = [
    "student_id",
    "wave",
    "study_hours",
    "attendance_rate",
    "quiz_score",
    "task_completion_rate",
]

student_missing_columns = [
    column
    for column in expected_student_columns
    if column not in students.columns
]
learning_missing_columns = [
    column
    for column in expected_learning_columns
    if column not in learning.columns
]
schema_checks = {
    "E10 必需字段齐全": not student_missing_columns,
    "E10 字段及顺序一致": students.columns.tolist() == expected_student_columns,
    "E11 必需字段齐全": not learning_missing_columns,
    "E11 字段及顺序一致": learning.columns.tolist() == expected_learning_columns,
}

for check_name, passed in schema_checks.items():
    print(f"{check_name}：{passed}")
if not all(schema_checks.values()):
    raise ValueError(
        "输入字段核验未通过。"
        f"E10 缺少字段：{student_missing_columns}；"
        f"E11 缺少字段：{learning_missing_columns}。"
        "请修正个人成果，或把对应输入开关改为 False。"
    )

student_checks = {
    "E10 记录数为 160": len(students) == 160,
    "E10 student_id 唯一": students["student_id"].is_unique,
    "E10 alias 唯一": students["alias"].is_unique,
    "E10 学习时长缺失数为 8": students["reported_study_hours"].isna().sum() == 8,
    "E10 出勤率缺失数为 4": students["reported_attendance_rate"].isna().sum() == 4,
    "E10 gender 取值正确": set(students["gender"]) == {"female", "male"},
    "E10 major 取值正确": set(students["major"]) == {
        "data_science",
        "applied_statistics",
        "statistical_computing",
    },
    "E10 编程经历取值正确": set(students["prior_programming"]) == {"yes", "no"},
    "E10 支持方案取值正确": set(students["support_plan"]) == {"regular", "feedback"},
    "E10 学习时长范围合理": numeric_range_ok(
        students,
        "reported_study_hours",
        0,
        40,
    ),
    "E10 出勤率范围合理": numeric_range_ok(
        students,
        "reported_attendance_rate",
        0,
        100,
    ),
}
records_per_student = learning.groupby("student_id")["wave"].size()
learning_checks = {
    "E11 记录数为 640": len(learning) == 640,
    "E11 学生数为 160": learning["student_id"].nunique() == 160,
    "E11 每名学生有 4 轮记录": records_per_student.eq(4).all(),
    "E11 四个轮次齐全": set(learning["wave"]) == {1, 2, 3, 4},
    "E11 复合主键唯一": not learning.duplicated(
        ["student_id", "wave"]
    ).any(),
    "E11 出勤率缺失数为 4": learning["attendance_rate"].isna().sum() == 4,
    "E11 测验成绩缺失数为 11": learning["quiz_score"].isna().sum() == 11,
    "E11 任务完成率缺失数为 5": learning["task_completion_rate"].isna().sum() == 5,
    "E11 学习时长范围合理": numeric_range_ok(
        learning,
        "study_hours",
        0,
        40,
    ),
    "E11 出勤率范围合理": numeric_range_ok(
        learning,
        "attendance_rate",
        0,
        100,
    ),
    "E11 测验成绩范围合理": numeric_range_ok(
        learning,
        "quiz_score",
        0,
        100,
    ),
    "E11 任务完成率范围合理": numeric_range_ok(
        learning,
        "task_completion_rate",
        0,
        100,
    ),
}

input_checks = {**student_checks, **learning_checks}
for check_name, passed in input_checks.items():
    print(f"{check_name}：{passed}")
if not all(input_checks.values()):
    failed_checks = [
        check_name
        for check_name, passed in input_checks.items()
        if not passed
    ]
    raise ValueError(
        "输入口径核验未通过："
        + "、".join(failed_checks)
        + "。请修正个人成果，或把对应输入开关改为 False。"
    )

# 下载的官方检查点通过同一套输入核验后才保存。
for cache_path, data in downloaded_checkpoints.items():
    save_download(cache_path, data)

project_path = local_or_download(project_path, project_path.name)
outcome_path = local_or_download(outcome_path, outcome_path.name)

## 任务 1：使用 `concat()` 纵向拼接同结构记录

课程代码已经按四个轮次准备好同结构子表，并提供复合主键核验。请使用 `pd.concat()` 把它们按行纵向拼接，再运行课程检查代码。这个任务只要求掌握 `concat()` 的核心写法，不要求独立编写 `itertuples()` 等核验细节。

In [ ]:
# 课程提供：按轮次准备四张同结构子表
wave_parts = [
    learning.loc[learning["wave"] == wave].copy()
    for wave in [1, 2, 3, 4]
]

# TODO：纵向拼接四个轮次，再用课程提供的代码核验复合主键
# recombined_learning = (
#     pd.concat(..., ignore_index=True)
#     .sort_values(["student_id", "wave"])
#     .reset_index(drop=True)
# )

# 课程提供：补全拼接后取消注释，核对复合主键
# original_keys = list(
#     learning.sort_values(["student_id", "wave"])[
#         ["student_id", "wave"]
#     ].itertuples(index=False, name=None)
# )
# recombined_keys = list(
#     recombined_learning[["student_id", "wave"]].itertuples(
#         index=False,
#         name=None,
#     )
# )

# print(f"各部分行数：{[len(part) for part in wave_parts]}")
# print(f"拼接后行数：{len(recombined_learning)}")
# print(f"复合主键顺序一致：{original_keys == recombined_keys}")

## 任务 2：运行课程代码整理并检查项目成绩表

项目成绩表包含 `student_id + wave` 复合主键。下面的课程代码复用 E10 已学的文本与数值清理，然后报告记录数、主键重复和成绩缺失。本任务重点是确认项目表的行代表什么，不要求再次从零编写清洗流程。

In [ ]:
# 课程提供的项目表清洗与检查代码：直接运行并解释结果；无需重写。
projects = pd.read_csv(project_path)
projects["student_id"] = (
    projects["student_id"].astype("string").str.strip()
)
projects["project_name"] = (
    projects["project_name"].astype("string").str.strip()
)
projects["wave"] = pd.to_numeric(
    projects["wave"],
    errors="coerce",
).astype("Int64")
projects["project_score"] = pd.to_numeric(
    projects["project_score"],
    errors="coerce",
)
projects = projects.sort_values(
    ["student_id", "wave"]
).reset_index(drop=True)

print(f"项目记录数：{len(projects)}")
print(f"复合主键重复数：{projects.duplicated(['student_id', 'wave']).sum()}")
print(f"项目成绩缺失数：{projects['project_score'].isna().sum()}")

## 任务 3：比较 `inner` 与 `left` 并检查匹配

以四轮学习记录为左表，按 `student_id + wave` 分别完成：

- `left` 连接：保留全部 640 条学习记录；
- `inner` 连接：只保留项目表中能够匹配的组合。

课程代码已经提供 `validate`、`indicator` 和匹配状态核验；请补全两种连接方式并解释行数差异。未匹配记录与已匹配但成绩缺失具有不同含义。

In [ ]:
# 课程提供：学生学号与轮次组成复合主键
merge_keys = ["student_id", "wave"]

# TODO：补全两种连接方式，再用课程提供的代码核对匹配状态
# learning_projects = learning.merge(
#     projects,
#     on=merge_keys,
#     how=...,
#     validate="one_to_one",
#     indicator=True,
# )
# inner_learning_projects = learning.merge(
#     projects,
#     on=merge_keys,
#     how=...,
#     validate="one_to_one",
# )

# 课程提供：补全两种连接后取消注释，核对匹配状态
# unmatched_projects = (
#     learning_projects["_merge"] == "left_only"
# ).sum()
# matched_missing_scores = (
#     (learning_projects["_merge"] == "both")
#     & learning_projects["project_score"].isna()
# ).sum()

# print(f"left 连接行数：{len(learning_projects)}")
# print(f"inner 连接行数：{len(inner_learning_projects)}")
# print(f"没有项目记录的组合：{unmatched_projects}")
# print(f"已匹配但项目成绩缺失：{matched_missing_scores}")
# print(learning_projects["_merge"].value_counts())

## 任务 4：构造学生层学习和项目汇总

使用命名聚合，把一名学生的多条记录汇总成一行，以 `student_id` 分组并保留为普通列：

| 汇总表 | 输出列 | 来源字段与统计量 |
| --- | --- | --- |
| `learning_summary` | `study_hours` | 学习记录中 `study_hours` 的均值 |
| `learning_summary` | `attendance_rate` | 学习记录中 `attendance_rate` 的均值 |
| `learning_summary` | `task_completion_rate` | 学习记录中 `task_completion_rate` 的均值 |
| `project_summary` | `submitted_projects` | 项目表中 `project_score` 的非缺失数量（`count`） |
| `project_summary` | `mean_project_score` | 项目表中 `project_score` 的均值 |

均值使用对应字段的非缺失值，并四舍五入到一位小数。`submitted_projects` 表示有有效成绩的项目数，保留整数含义。

学习均值汇总完成后，运行课程提供的轮次提取和合并代码，为 `learning_summary` 加入 `first_quiz_score`（第 1 轮）和 `last_quiz_score`（第 4 轮）。原轮次成绩缺失时继续保留缺失。

输出两张汇总表的形状，以及第 1、4 轮成绩各自的缺失数。

In [ ]:
# TODO：补全学生层学习汇总和项目汇总的聚合规则
# learning_summary = (
#     learning.groupby("student_id", as_index=False)
#     .agg(
#         study_hours=...,
#         attendance_rate=...,
#         task_completion_rate=...,
#     )
#     .round(1)
# )

# 课程提供：学习汇总完成后取消注释，提取第 1、4 轮成绩并保留缺失
# quiz_rounds = (
#     learning.loc[
#         learning["wave"].isin([1, 4]),
#         ["student_id", "wave", "quiz_score"],
#     ]
#     .pivot(
#         index="student_id",
#         columns="wave",
#         values="quiz_score",
#     )
#     .rename(
#         columns={
#             1: "first_quiz_score",
#             4: "last_quiz_score",
#         }
#     )
#     .reset_index()
# )
# learning_summary = learning_summary.merge(
#     quiz_rounds,
#     on="student_id",
#     how="left",
#     validate="one_to_one",
# )

# project_summary = (
#     projects.groupby("student_id", as_index=False)
#     .agg(
#         submitted_projects=...,
#         mean_project_score=...,
#     )
#     .round(1)
# )

# print(f"学习汇总 shape：{learning_summary.shape}")
# print(f"项目汇总 shape：{project_summary.shape}")
# print(f"第 1 轮成绩缺失数：{learning_summary['first_quiz_score'].isna().sum()}")
# print(f"第 4 轮成绩缺失数：{learning_summary['last_quiz_score'].isna().sum()}")

## 任务 5：合并学生主表、汇总表和课程结果

课程代码已经完成课程结果表的基础类型检查。请以清洗后的学生调查表为主体，按 `student_id` 依次连接学习汇总、项目汇总和课程结果。单键、左连接和一对一关系已经写入脚手架，你只需补全三张右表。

In [ ]:
# 课程提供：读取课程结果、统一字段类型并选出学生主表字段
outcomes = pd.read_csv(outcome_path)
outcomes["student_id"] = (
    outcomes["student_id"].astype("string").str.strip()
)
for column in [
    "baseline_score",
    "final_score",
    "data_literacy_score",
]:
    outcomes[column] = pd.to_numeric(
        outcomes[column],
        errors="coerce",
    )

student_columns = [
    "student_id",
    "alias",
    "gender",
    "major",
    "prior_programming",
    "support_plan",
]

# TODO：补全依次连接的三张表，检查最终行列数
# analysis_ready = (
#     students[student_columns]
#     .merge(
#         ...,
#         on="student_id",
#         how="left",
#         validate="one_to_one",
#     )
#     .merge(
#         ...,
#         on="student_id",
#         how="left",
#         validate="one_to_one",
#     )
#     .merge(
#         ...,
#         on="student_id",
#         how="left",
#         validate="one_to_one",
#     )
#     .sort_values("student_id")
#     .reset_index(drop=True)
# )

# print(f"最终行列数：{analysis_ready.shape}")
# print(analysis_ready.head())

## 任务 6：最终核验并导出分析就绪表

先完成任务 5，得到 `analysis_ready`，再运行课程提供的核验代码。检查内容包括 160 名唯一学生、`expected_columns` 指定的字段顺序、成绩及项目数范围，以及预期缺失：

- `first_quiz_score` 保留 5 个原始缺失。
- `last_quiz_score` 保留 3 个原始缺失。
- 其余正式字段均无缺失。

检查全部通过后，将 `analysis_ready` 导出到准备单元给出的 `output_path`，格式为 CSV，使用 `index=False`，并输出保存路径。该文件供第 5、6 章实验使用。

In [ ]:
# 课程提供：正式字段顺序及应当完整的字段
expected_columns = [
    "student_id",
    "alias",
    "gender",
    "major",
    "prior_programming",
    "support_plan",
    "study_hours",
    "attendance_rate",
    "task_completion_rate",
    "first_quiz_score",
    "last_quiz_score",
    "submitted_projects",
    "mean_project_score",
    "baseline_score",
    "final_score",
    "data_literacy_score",
]
complete_columns = [
    column
    for column in expected_columns
    if column not in {"first_quiz_score", "last_quiz_score"}
]

# 课程提供：核验上一任务得到的分析就绪表
checks = {
    "记录数为 160": len(analysis_ready) == 160,
    "student_id 唯一": analysis_ready["student_id"].is_unique,
    "字段顺序正确": analysis_ready.columns.tolist() == expected_columns,
    "第 1 轮成绩缺失数为 5": analysis_ready["first_quiz_score"].isna().sum() == 5,
    "第 4 轮成绩缺失数为 3": analysis_ready["last_quiz_score"].isna().sum() == 3,
    "其余正式字段没有缺失": not analysis_ready[complete_columns].isna().any().any(),
    "期末成绩范围合理": analysis_ready["final_score"].between(0, 100).all(),
    "项目数范围合理": analysis_ready["submitted_projects"].between(0, 4).all(),
}

for check_name, passed in checks.items():
    print(f"{check_name}：{passed}")
if not all(checks.values()):
    raise ValueError("分析就绪表核验未通过，请返回前面的任务检查。")

output_path.parent.mkdir(parents=True, exist_ok=True)
# TODO：将核验通过的分析就绪表导出为 CSV
# ...
# print(f"已导出：{output_path}")

## 任务 7：数据范围说明与实验小结

用 Markdown 回答：

1. 为什么 `left` 连接得到 640 行，而 `inner` 连接得到 620 行？
2. 未匹配项目记录和已匹配但项目成绩缺失分别表示什么？
3. 为什么第一轮、第四轮成绩中的原始缺失必须继续保留？
4. 为什么最终分析表仍为 160 行，后续能够描述什么、不能直接证明什么？

### 数据范围说明与实验小结

- 两种连接的行数：
- 两类项目缺失：
- 保留首末轮成绩缺失的原因：
- 最终分析表的学生覆盖：